# Fields and Aggregations

In SLayer, a **field** is a named row-level SQL expression. The **aggregation** (sum, avg, count, etc.) is specified at query time as a function call: `sum(revenue)`, `count(*)`, `weighted_avg(tax_rate, weight=subtotal)`.

This notebook demonstrates the full range of aggregation features with working code.

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Basic Aggregations

Every metric expression specifies both the field and the aggregation as a function call. `count(*)` is the universal row count.

In [2]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(*)", "sum(order_total)", "avg(order_total)", "min(order_total)", "max(order_total)"],
    }
)

pd.DataFrame(result.data)

,orders._count,orders.order_total_sum,orders.order_total_avg,orders.order_total_min,orders.order_total_max
0,662572,7347543.53,11.09,0.00,106.42


## Inspecting Dataset Fields

Auto-ingested datasets have one field per underlying table column - no need for num_fields x num_aggregations predefined metrics, those can be done on the fly. 

In [3]:
orders_dataset = next(d for d in datasets if d.name == "orders")

print(f"orders_dataset has {len(orders_dataset.fields)} fields:")
for col in orders_dataset.fields:
    sql_str = col.sql or col.name
    print(f"  {col.name:<20} sql: {sql_str:<20} datatype: {col.datatype}")

print("\nAny field can be aggregated at query time via a function call,")
print("subject to the field's datatype and allowed_aggregations whitelist.")
print("A sole primary key defaults to count / count_distinct / min / max only.")
print("Plus count(*) is always available for COUNT(*).")


orders_dataset has 9 fields:
  id                   sql: id                   datatype: String
  customer_id          sql: customer_id          datatype: String
  ordered_at           sql: ordered_at           datatype: Date
  store_id             sql: store_id             datatype: String
  subtotal             sql: subtotal             datatype: Decimal
  tax_paid             sql: tax_paid             datatype: Decimal
  order_total          sql: order_total          datatype: Decimal
  status               sql: status               datatype: String
  item_skus            sql: item_skus            datatype: Array

Any field can be aggregated at query time via a function call,
subject to the field's datatype and allowed_aggregations whitelist.
A sole primary key defaults to count / count_distinct / min / max only.
Plus count(*) is always available for COUNT(*).


## Multiple Aggregations on One Field

One field, many aggregations — no need to define `order_total_sum`, `order_total_avg`, etc. as separate metrics.

In [4]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(*)", "sum(order_total)", "avg(order_total)"],
        "dimensions": ["stores.name"],
        "order": [{"field": "sum(order_total)", "direction": "desc"}],
        "limit": 5,
    }
)

pd.DataFrame(result.data)

,orders.stores.name,orders._count,orders.order_total_sum,orders.order_total_avg
0,Brooklyn,256909,2805157.85,10.92
1,Philadelphia,197665,2135270.98,10.80
2,Chicago,102763,1185799.34,11.54
3,San Francisco,89967,1046570.77,11.63
4,New Orleans,15268,174744.59,11.45


## Arithmetic on Aggregated Metrics

Aggregated metrics compose naturally in arithmetic expressions.

In [5]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": [
            "count(*)",
            "sum(order_total)",
            {"metric": "sum(order_total) / count(*)", "alias": "aov", "label": "Avg Order Value"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"field": "sum(order_total)", "direction": "desc"}],
        "limit": 5,
    }
)

pd.DataFrame(result.data)

,orders.stores.name,orders._count,orders.order_total_sum,orders.aov
0,Brooklyn,256909,2805157.85,10.92
1,Philadelphia,197665,2135270.98,10.80
2,Chicago,102763,1185799.34,11.54
3,San Francisco,89967,1046570.77,11.63
4,New Orleans,15268,174744.59,11.45


## Share of Parent (`partition_by`)

Most aggregations take an optional `partition_by=` to compute over a **subset** of the query's dimensions, repeated across the finer rows — like `SUM(x) OVER (PARTITION BY ...)`. `partition_by=[]` is the grand total (share of total); a dimension, list, or dotted path gives share-of-parent within a group. The total is computed over rows passing row-level filters, so `having`-style filters and pagination never change it.

It composes with `window=`, `first`/`last`, transforms, and filter / `order` references — cross-dataset included, e.g. `last(customers.spend, partition_by=customers.tier)` or a cross-dataset partitioned aggregate nested inside a transform.

In [6]:
# `partition_by=` computes an aggregation at a COARSER grain than the query and
# repeats it across the finer rows. `partition_by=[]` is the grand total, so
# each store's share of total revenue sums to 100%. A dimension, list, or dotted
# path (e.g. partition_by=region) gives share-of-parent within a group.
result = engine.execute_sync(
    query={
        "population": "orders",
        "dimensions": ["stores.name"],
        "metrics": [
            {"metric": "sum(order_total)", "alias": "store_rev"},
            {"metric": "sum(order_total) / sum(order_total, partition_by=[])", "alias": "share_of_total"},
        ],
        "order": [{"field": "store_rev", "direction": "desc"}],
    }
)

df = pd.DataFrame(result.data)
print(f"sum of shares: {df['orders.share_of_total'].sum():.1%}")
df

sum of shares: 100.0%


,orders.stores.name,orders.store_rev,orders.share_of_total
0,Brooklyn,2805157.85,0.38
1,Philadelphia,2135270.98,0.29
2,Chicago,1185799.34,0.16
3,San Francisco,1046570.77,0.14
4,New Orleans,174744.59,0.02


## Expression Dimensions and Conditionals (`CASE WHEN` / `iif`)

Expressions for metrics, filters, and dimensions can branch with SQL `CASE WHEN c THEN x [WHEN ...] [ELSE y] END` — or the equivalent call form `iif(c, x, y)` — rendered as portable SQL `CASE` on every backend. Branch types must agree: a `NULL` branch (a missing `ELSE`) is absorbed by the other, a numeric mix widens to `DOUBLE`, and any other mix is a plan-time error.

A dimension can itself be a computed expression: pass `{"expression": ..., "name": ...}` (or just a non-identifier string) in `dimensions` and the query groups by the computed value. Its name works in `order` and `filter` like any other dimension.

In [7]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "dimensions": [
            {
                "field": "CASE WHEN order_total >= 15 THEN 'large' ELSE 'small' END",
                "alias": "size",
            },
        ],
        "metrics": [
            {"metric": "count(*)", "alias": "n_orders"},
            {"metric": "sum(order_total)", "alias": "revenue"},
            {"metric": "avg(order_total)", "alias": "average_order_size"},
        ],
        "order": [{"field": "revenue", "direction": "desc"}],
    }
)

pd.DataFrame(result.data)

,orders.size,orders.n_orders,orders.revenue,orders.average_order_size
0,large,151266,4122421.44,27.25
1,small,511306,3225122.09,6.31


## Transforms on Aggregated Metrics

All transform functions (`cumsum`, `change`, `time_shift`, `rank`, etc.) wrap aggregated metric refs.

In [8]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": [
            "sum(order_total)",
            {"metric": "cumsum(sum(order_total))", "alias": "running_total"},
            {"metric": "change(sum(order_total))", "alias": "mom_change"},
        ],
        "time_dimensions": [{"field": "ordered_at", "granularity": "month"}],
        "order": [{"field": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

pd.DataFrame(result.data)

,orders.ordered_at,orders.order_total_sum,orders.running_total,orders.mom_change
0,2023-11-01,6476.29,6476.29,None
1,2023-12-01,22484.60,28960.89,16008.31
2,2024-01-01,26483.61,55444.50,3999.01
3,2024-02-01,33301.17,88745.67,6817.56
4,2024-03-01,43013.17,131758.84,9712.00
5,2024-04-01,47661.36,179420.20,4648.19


## Cross-Dataset Aggregated Metrics

Cross-dataset metrics use dot syntax for the relationship path, wrapped in the aggregation. `count(customers.*)` means COUNT(*) on the joined customers dataset.

In [9]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": [
            "count(*)",
            "sum(order_total)",
            "count(customers.*)",
        ],
        "time_dimensions": [{"field": "ordered_at", "granularity": "month"}],
        "order": [{"field": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

pd.DataFrame(result.data)

,orders.ordered_at,orders._count,orders.order_total_sum,orders.customers._count
0,2023-11-01,491,6476.29,121
1,2023-12-01,2010,22484.60,162
2,2024-01-01,2282,26483.61,195
3,2024-02-01,2950,33301.17,246
4,2024-03-01,3925,43013.17,287
5,2024-04-01,4209,47661.36,318


## Custom Aggregation: Weighted Average

Let's define a dataset with a custom weighted average aggregation.

In [10]:
from slayer.core.enums import DataType
from slayer.core.models import Aggregation, AggregationParam, DatasetField, DatasetRelationship, SlayerDataset

orders_custom = SlayerDataset(
    name="orders_custom",
    sql_table="orders",
    data_source="jaffle_shop",
    fields=[
        DatasetField(name="id", sql="id", datatype=DataType.STRING, primary_key=True),
        DatasetField(name="store_id", sql="store_id", datatype=DataType.STRING),
        DatasetField(name="subtotal", sql="subtotal", datatype=DataType.FLOAT),
        DatasetField(name="store_tax_rate", sql="stores.tax_rate", datatype=DataType.FLOAT),
    ],
    relationships=[DatasetRelationship(to="stores", from_columns=["store_id"], to_columns=["id"])],
    aggregations=[
        Aggregation(
            name="weighted_avg",
            expression="SUM({value} * {weight}) / NULLIF(SUM({weight}), 0)",
            params=[AggregationParam(name="weight", sql="subtotal")], # sets the default for the weight parameter
            description="Average of value weighted by weight",
        ),
    ],
)
print(f"Custom aggregations on orders_custom: {[a.name for a in orders_custom.aggregations]}")


Custom aggregations on orders_custom: ['weighted_avg']


## Allowed Aggregations Whitelist

Restrict which aggregations make sense for a given field.

In [11]:
# Restrict per-field aggregation eligibility via `allowed_aggregations`. The
# whitelist overrides the datatype-default eligibility rule (numeric fields get
# sum/avg/min/max/etc by default; PK fields are always count-only).
from slayer.core.enums import DataType
from slayer.core.models import DatasetField, SlayerDataset

restricted_dataset = SlayerDataset(
    name="restricted_orders",
    sql_table="orders",
    data_source="jaffle_shop",
    fields=[
        DatasetField(name="store_id", sql="store_id", datatype=DataType.STRING),
        DatasetField(
            name="subtotal", sql="subtotal", datatype=DataType.FLOAT,
            allowed_aggregations=["sum", "avg"],
        ),
    ],
)
sub_col = next(c for c in restricted_dataset.fields if c.name == "subtotal")
print(f"subtotal field allows: {sub_col.allowed_aggregations}")


subtotal field allows: ['sum', 'avg']


## Generated SQL

The aggregation syntax compiles to standard SQL aggregation functions.

In [12]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(*)", "sum(order_total)", "avg(order_total)"],
        "dimensions": ["stores.name"],
        "limit": 3,
    }
)

print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  stores.name AS "orders.stores.name",
  COUNT(*) AS "orders._count",
  SUM(orders.order_total) AS "orders.order_total_sum",
  AVG(orders.order_total) AS "orders.order_total_avg"
FROM orders AS orders
LEFT JOIN stores AS stores
  ON orders.store_id = stores.id
GROUP BY
  stores.name
ORDER BY
  "orders.stores.name" ASC
LIMIT 3


## Summary

| Concept | Syntax | Example |
|---------|--------|---------|
| Aggregation at query time | `agg(field)` | `sum(revenue)`, `avg(price)` |
| Row count | `count(*)` | Always available, no metric needed |
| Non-null count | `count(col)` | `count(email)` |
| Distinct count | `count_distinct(col)` | `count_distinct(customer_id)` |
| First/last by time | `first(col)`, `last(col)` | `last(balance, updated_at)` |
| Weighted average | `weighted_avg(col, weight=w)` | `weighted_avg(tax_rate, weight=subtotal)` |
| Percentile | `percentile(col, p=0.95)` | `percentile(latency, p=0.99)` |
| Median | `median(col)` | Shorthand for percentile(p=0.5) |
| Share of parent | `agg(col, partition_by=...)` | `sum(revenue, partition_by=region)`, `partition_by=[]` for total |
| Custom expression | Define in dataset `aggregations` | `trimmed_mean(score, lo=10, hi=90)` |
| Whitelist | `allowed_aggregations` on field | Restricts valid aggregations |
| Conditional | `CASE WHEN ... END` / `iif(c, x, y)` | `CASE WHEN sum(revenue) >= 1000 THEN 'hi' ELSE 'lo' END` |
| Expression dimension | `{"expression": ..., "name": ...}` in `dimensions` | Groups by the computed value |

See the [aggregations guide](aggregations.md) for the full explanation.